In [1]:
from langchain_cerebras import ChatCerebras
from langchain_groq import ChatGroq
from langchain import chat_models
import os
from dotenv import load_dotenv
load_dotenv()
os.environ["CEREBRAS_API_KEY"]=os.getenv("CEREBRAS_API_KEY")
os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")


llm_gpt= ChatCerebras( model="gpt-oss-120b",
    api_key=os.getenv("CEREBRAS_API_KEY"))

llm_groq=ChatGroq(model="llama-3.1-8b-instant",
api_key=os.getenv("GROQ_API_KEY"))

#response_gpt =llm_gpt.invoke("Hi mate!")
#response_groq =llm_groq.invoke("Hi mate!")
#print(response_gpt.content)
#print(response_groq.content)

In [2]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

prompt=ChatPromptTemplate.from_messages([
    ("system","You are a fan boy of {character}"),
    ("human","hey dude, create detailed para in {words_count} words about {character} on this comeback from rockbottom to success!")
])

chain_gpt= prompt | llm_gpt
chain_groq=prompt | llm_groq

#for chunk in chain.stream({"character":"batman","words_count":"100" }):
 #   print(chunk.content,end="", flush=True)

batch_inputs =[
{"character":"batman","words_count":"100" },
{"character":"walter white","words_count":"100"} ,
]

#responses = chain_groq.batch(batch_inputs)
#for response in responses:
#    print(response.content)

In [3]:
from langchain_core.tools import tool
from langchain_core.messages import ToolMessage

@tool
def sendMail(sender: str, receiver: str, sub: str, body: str) -> str:
    """Send the mail to the receiver."""

    print("\n========== EMAIL ==========")
    print(f"From    : {sender}")
    print(f"To      : {receiver}")
    print(f"Subject : {sub}")
    print("---------------------------")
    print(body)
    print("===========================\n")

    return f"Mail sent successfully to {receiver}"

llm_with_tools = llm_groq.bind_tools([sendMail])

message=[

{"role": "user",
        "content": """
Sender: fightclub@gmail.com
Receiver: itslokeshx@gmail.com
Subject: Don't talk about Fight Club

Write a welcome email from Tyler Durden.
Start with "Welcome to Fight Club".
Mention the first two rules.
Then send the email using the tool.
"""}

]

AI_return_response = llm_with_tools.invoke(message)
AI_return_response.tool_calls

message.append(AI_return_response)

In [4]:
for tool_call in AI_return_response.tool_calls:
    tool_result = sendMail.invoke(tool_call["args"])
    message.append(    ToolMessage(
        content=tool_result,
        tool_call_id=tool_call["id"]
    )
    )
print(AI_return_response.tool_calls)



========== EMAIL ==========
From    : fightclub@gmail.com
To      : itslokeshx@gmail.com
Subject : Don't talk about Fight Club
---------------------------
Welcome to Fight Club. You'll shoot a bullet with your bare hands. No women. Ever.


========== EMAIL ==========
From    : fightclub@gmail.com
To      : itslokeshx@gmail.com
Subject : Don't talk about Fight Club
---------------------------
The first rule of Fight Club is: you do not talk about Fight Club. The second rule of Fight Club is: you DO NOT TALK ABOUT FIGHT CLUB.

[{'name': 'sendMail', 'args': {'body': "Welcome to Fight Club. You'll shoot a bullet with your bare hands. No women. Ever.", 'receiver': 'itslokeshx@gmail.com', 'sender': 'fightclub@gmail.com', 'sub': "Don't talk about Fight Club"}, 'id': '36gcep3sb', 'type': 'tool_call'}, {'name': 'sendMail', 'args': {'body': 'The first rule of Fight Club is: you do not talk about Fight Club. The second rule of Fight Club is: you DO NOT TALK ABOUT FIGHT CLUB.', 'receiver': 'itslo

In [5]:
result=llm_with_tools.invoke(message)
result.content

'Please note that this is a text representation of the email and is not an actual email. Also, the email content is a reference to the movie Fight Club and is not intended to be taken literally.'